# 08 · Authoring a bundle

## Goal

Build the real skill bundle: one routing skill plus three sub-skills for the
Contract Renewal Desk, deliberately reproduce two selection failure modes
(overlapping descriptions, silent non-selection), fix them, and use the
GHCP sandbox to ship one skill with an executable Python script.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
from csx.clients import get_copilot_client
settings = load_settings()
client = get_copilot_client(settings, delegated=True)
from pathlib import Path
assert (Path("../skills/negotiation-playbook/SKILL.md")).exists(), "run 07 first"


## Concept

A routing skill's job is entirely its `description` field — the selector
reads descriptions, not bodies, to decide what to load. That makes skill
authoring mostly a precision-writing exercise, and it fails in two
characteristic ways worth seeing on purpose:

- **Overlapping descriptions**: two skills whose descriptions both plausibly
  match a prompt: the selector picks one, unpredictably, and the other's
  capability silently doesn't fire.
- **Silent non-selection**: a description too narrow or too jargon-heavy for
  the way users actually phrase the request — the skill exists, is correct,
  and never gets picked.

Both look identical from the outside: "the agent didn't do the thing." The
fix is always the same lever as `02` and `06` — better descriptions — just
applied to a different surface, which is the throughline of findings #3
across this whole curriculum.


## Build


### The bundle: one router, three sub-skills


In [ ]:
from pathlib import Path
skills_dir = Path("../agents/contract-renewal-desk/skills")
skills_dir.mkdir(exist_ok=True)

bundle = {
    "renewal-routing": (
        "Routes a renewal request to the right sub-skill based on urgency and spend signals. "
        "Use this whenever a user asks to handle, process, or decide on a specific supplier renewal.",
        "# Renewal routing\n\nDecide: routine-renewal, escalation, or negotiation-drafting, then hand off.\n"
    ),
    "routine-renewal": (
        "Handles a renewal with flat spend and no performance flags — confirm terms and proceed.",
        "# Routine renewal\n\nConfirm notice period and pricing are unchanged, recommend renew.\n"
    ),
    # Deliberately overlapping with routine-renewal at first, on purpose —
    # fixed two cells down.
    "escalation-handling": (
        "Handles a renewal that needs review.",
        "# Escalation handling\n\nSummarise the spend/performance signals driving escalation, name an owner.\n"
    ),
    "negotiation-drafting": (
        "Drafts negotiation correspondence once escalation has been reviewed by a human.",
        "# Negotiation drafting\n\nUse skills/../skills/negotiation-playbook — do not duplicate its content here.\n"
    ),
}

for name, (description, body) in bundle.items():
    d = skills_dir / name
    d.mkdir(exist_ok=True)
    (d / "SKILL.md").write_text(f"---\nname: {name}\ndescription: {description}\n---\n\n{body}")
print("bundle written:", list(bundle.keys()))


### Reproduce overlapping-description failure on purpose


In [ ]:
from csx.pac import copilot_push
import subprocess
copilot_push(Path('../agents/contract-renewal-desk'))
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)

ambiguous = client.ask_question("Handle the Northwind Fasteners renewal — spend is up 40%.")
print(ambiguous.text)
print("Look for which skill actually fired via the transcript (notebook 23 gives you the tooling to see this precisely).")


### Fix: make escalation-handling's description unambiguous, not just longer


In [ ]:
d = skills_dir / "escalation-handling"
text = (d / "SKILL.md").read_text()
text = text.replace(
    "Handles a renewal that needs review.",
    "Handles a renewal where spend has increased materially or performance has degraded (late deliveries, quality flags) — distinct from routine-renewal, which is for flat-spend, no-flag cases."
)
(d / "SKILL.md").write_text(text)
copilot_push(Path('../agents/contract-renewal-desk'))
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


### One skill ships a sandbox script


In [ ]:
script_dir = skills_dir / "routine-renewal"
(script_dir / "compute_uplift.py").write_text(
    "def recommended_uplift(spend_delta_pct: float) -> float:\n"
    "    # Simple bounded uplift recommendation the agent's sandbox can execute directly.\n"
    "    return max(0.0, min(spend_delta_pct * 0.1, 5.0))\n"
)
print("routine-renewal skill now ships an executable helper the GHCP sandbox runs on demand")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))
cases = load_golden(tags=["core"]) + load_golden(tags=["workflow"])  # workflow-tagged cases exercise routing language ahead of 09
suite = run_suite(client, cases=[c for c in cases if c["id"].startswith("core") or c["id"] == "wf-02-escalate-branch"], credit_meter=meter, min_pass_rate=0.8)


## Cost


In [ ]:
meter.report_cost("08", budget=settings.get("COPILOT_CREDIT_BUDGET"), delta_credits=suite.total_credits, note="bundle authoring + two selection-failure repros + fix")


## Teardown


In [ ]:
print("No teardown — skill bundle persists for the rest of the curriculum.")
